# Full-song inference notebook

This notebook loads one test song from the chunked WAV + mel datasets, runs chunk-wise separation, reconstructs each predicted mel chunk to audio, and concatenates the chunks into full-song stem WAV files.

In [1]:
from pathlib import Path
import sys
import torch
import torchaudio
import matplotlib.pyplot as plt
from IPython.display import Audio, display

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from dataset.mel_dataset import MelDatasetConfig
from model.separation_unet import SeparationUNet, SeparationUNetConfig
from processing.audio_transforms import MelSpecConfig, MelSpectrogramDenormalizer, MelSpectrogramToWav, save_audio


In [3]:
WAV_ROOT = Path('../musdb18/wavs')
MEL_ROOT = Path('../musdb18/mels')
CHECKPOINT_PATH = Path('../train/checkpoints/best.pt')
OUTPUT_DIR = Path('inference_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

SPLIT = 'test'
SONG_NAME = 'Louis Cressy Band - Good Time'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

mel_config = MelSpecConfig(
    sample_rate=22050,
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    n_mels=128,
    f_min=30.0,
    f_max=11025.0,
    top_db=80.0,
)
denormalizer = MelSpectrogramDenormalizer(min_db=-80.0, max_db=0.0)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
TARGET_STEMS = tuple(checkpoint.get('target_stems', ('drums', 'bass', 'other', 'vocals')))
model = SeparationUNet(SeparationUNetConfig(out_stems=len(TARGET_STEMS), stem_names=TARGET_STEMS)).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

song_wav_dir = WAV_ROOT / SPLIT / SONG_NAME
song_mel_dir = MEL_ROOT / SPLIT / SONG_NAME
chunk_dirs = sorted([p for p in song_mel_dir.iterdir() if p.is_dir() and p.name.startswith('chunk_')])
assert chunk_dirs, f'No chunk dirs found in {song_mel_dir}'
print('num chunks:', len(chunk_dirs))
print('target stems:', TARGET_STEMS)

num chunks: 29
target stems: ('drums', 'bass', 'other', 'vocals')


In [4]:
def load_mel_tensor(path: Path) -> torch.Tensor:
    x = torch.load(path, map_location='cpu')
    if x.dim() == 2:
        x = x.unsqueeze(0)
    return x.float()

predicted_song_audio = {stem: [] for stem in TARGET_STEMS}
target_song_audio = {stem: [] for stem in TARGET_STEMS}
mixture_song_audio = []

for chunk_dir in chunk_dirs:
    mel_chunk = load_mel_tensor(chunk_dir / 'mixture.pt').unsqueeze(0).to(DEVICE)
    mixture_wav, sr = torchaudio.load(str(song_wav_dir / chunk_dir.name / 'mixture.wav'))
    mixture_song_audio.append(mixture_wav)

    with torch.no_grad():
        pred_norm = model(mel_chunk).cpu().squeeze(0)

    for stem_idx, stem_name in enumerate(TARGET_STEMS):
        pred_db = denormalizer(pred_norm[stem_idx:stem_idx+1])
        reconstructor = MelSpectrogramToWav(mel_config, n_iter=32, length=mixture_wav.shape[-1])
        pred_wav = reconstructor(pred_db).cpu()
        predicted_song_audio[stem_name].append(pred_wav)

        target_wav, _ = torchaudio.load(str(song_wav_dir / chunk_dir.name / f'{stem_name}.wav'))
        target_song_audio[stem_name].append(target_wav)


/home/matej/Documents/music_segmentation/.venv/lib64/python3.11/site-packages/torchaudio/functional/functional.py:318: UserWarning: The length of signal is shorter than the length parameter. Result is being padded with zeros in the tail. Please check your center and hop_length settings. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:1199.)
  inverse = torch.istft(


RuntimeError: The size of tensor a (862) must match the size of tensor b (1723) at non-singleton dimension 2

In [ ]:
song_out_dir = OUTPUT_DIR / SONG_NAME
song_out_dir.mkdir(parents=True, exist_ok=True)

full_mixture = torch.cat(mixture_song_audio, dim=-1)
save_audio(song_out_dir / 'mixture_full.wav', full_mixture, mel_config.sample_rate)

for stem_name in TARGET_STEMS:
    pred_full = torch.cat(predicted_song_audio[stem_name], dim=-1)
    target_full = torch.cat(target_song_audio[stem_name], dim=-1)
    save_audio(song_out_dir / f'pred_{stem_name}_full.wav', pred_full, mel_config.sample_rate)
    save_audio(song_out_dir / f'target_{stem_name}_full.wav', target_full, mel_config.sample_rate)

print('saved outputs to:', song_out_dir)

In [ ]:
example_stem = TARGET_STEMS[0]
example_chunk = chunk_dirs[0]
mixture_mel = load_mel_tensor(example_chunk / 'mixture.pt')
target_mel = load_mel_tensor(example_chunk / f'{example_stem}.pt')

with torch.no_grad():
    pred_mel = model(mixture_mel.unsqueeze(0).to(DEVICE)).cpu()[0, 0:1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(mixture_mel[0], origin='lower', aspect='auto')
axes[0].set_title('Mixture mel')
axes[1].imshow(target_mel[0], origin='lower', aspect='auto')
axes[1].set_title(f'Target mel: {example_stem}')
axes[2].imshow(pred_mel[0], origin='lower', aspect='auto')
axes[2].set_title(f'Pred mel: {example_stem}')
plt.tight_layout()
plt.show()

In [ ]:
display(Audio(str(song_out_dir / 'mixture_full.wav')))
for stem_name in TARGET_STEMS:
    print(stem_name)
    display(Audio(str(song_out_dir / f'pred_{stem_name}_full.wav')))